In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# phasor representation diagram
class Phasor:

    def __init__(self):

        self.signals = []

        # plot formatting
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(1, 2, figsize=(10,5))

        # time (ms) to include 2 full 50 Hz waves
        self.t = np.linspace(0, 40, 300)

        # artists lists
        self.phasors = []
        self.projs   = []
        self.dots    = []
        self.waves   = []

    # add signal data to signals list
    def add_signal(self, A, omega, phi):
        self.signals.append([A, omega, phi])

    # build static plot elements
    def _build_axes(self):

        ax_phasor, ax_wave = self.ax

        # plot formatting
        self.fig.subplots_adjust(
            left=0.05, right=0.98,
            bottom=0.1, top=0.85,
            wspace=0.2, hspace=0.2
        )
        
        self.fig.suptitle(
            r"Phasor Representation of Sinusoids "
            r"$A \sin(\omega t + \phi)$", 
            fontsize=18
        )

        ax_phasor.set_aspect("equal")
        ax_phasor.set_title("Phasor in Complex Plane", fontsize=15)
        ax_phasor.set_xlabel(r"$\text{Re}$", fontsize=12)
        ax_phasor.set_ylabel(r"$\text{Im}$", fontsize=12)
        ax_phasor.axhline(0, linewidth=2, color="white", alpha=0.5)
        ax_phasor.axvline(0, linewidth=2, color="white", alpha=0.5)
        ax_phasor.grid(True, which="major", alpha=0.2)
        
        ax_wave.set_title("Sinusoidal Signal", fontsize=15)
        ax_wave.set_xlabel(r"Time $t$ (ms)", fontsize=12)
        ax_wave.set_ylabel(r"Displacement $A(t)$", fontsize=12)
        ax_wave.axhline(0, linewidth=2, color="white", alpha=0.5)
        ax_wave.grid(True, which="major", alpha=0.2)
        ax_wave.set_xlim(0, 40) # time (ms) for 2 50 Hz waves

        # add axis limits based on maximum amplitude signal
        lim = 1.1 * max(
            [signal[0] for signal in self.signals], default=1
        )
        ax_phasor.set_xlim(-lim, lim)
        ax_phasor.set_ylim(-lim, lim)
        ax_wave.set_ylim(-lim, lim)

    # initialise dynamic plot elements
    def _init_artists(self):

        # colour map for signal colours
        colours = plt.cm.hsv(
            np.linspace(0, 1, len(self.signals), endpoint=False)
        )

        ax_phasor, ax_wave = self.ax

        # loop over signals and create artists
        for (A, omega, phi), col in zip(self.signals, colours):

            # phasor with -pi/2 phase correction
            phase = phi - np.pi / 2
            re = A * np.cos(phase)
            im = A * np.sin(phase)

            # phasor vector
            self.phasors.append(ax_phasor.quiver(
                0, 0, re, im,
                color=col,
                angles="xy",
                scale_units="xy",
                scale=1,
                width=0.01,
                zorder=10
            ))
            
            # real projection
            self.projs.append(ax_phasor.plot(
                (re, re), (0, im),
                color=col,
                linewidth=1,
                linestyle="--"
            )[0])

            # real value
            self.dots.append(ax_phasor.scatter(
                re, 0,
                color=col,
                s=20,
                zorder=100
            ))

            # sinusoid
            y = A * np.sin(omega * self.t * 1e-3 + phi)
            self.waves.append(ax_wave.plot(
                self.t, y, 
                color=col, 
                linewidth=2
            )[0])

    # update artists for progressed time dt
    def _update(self, dt):

        for i, (A, omega, phi) in enumerate(self.signals):

            # update phase
            phase = omega * dt * 1e-3 + phi - np.pi / 2
            re = A * np.cos(phase)
            im = A * np.sin(phase)

            # update phasor vector
            self.phasors[i].set_offsets([[0, 0]])
            self.phasors[i].set_UVC(re, im)

            # update real projection
            self.projs[i].set_data(
                (re, re), (0, im)
            )

            # update real value
            self.dots[i].set_offsets([[re, 0]])

            # update sinusoid
            y = A * np.sin(omega * (self.t + dt) * 1e-3 + phi)
            self.waves[i].set_ydata(y)

    # plot all current signals
    def plot(self, dt=0):
        self._build_axes()
        self._init_artists()
        self._update(dt)
        self.fig.canvas.draw_idle()
        
    # animate all current signals
    def animate(self, frames, save=False, title="phasor"):

        self._build_axes()
        self._init_artists()

        # loop animation over 2 50 Hz cycles
        dt = np.linspace(0, 40, frames, endpoint=False)

        def update(frame):
            self._update(dt[frame])
            return (
                self.phasors + 
                self.projs + 
                self.dots + 
                self.waves
            )
                    
        anim = FuncAnimation(
            self.fig,
            update,
            frames=frames, 
            blit=True
        )

        # save at 10 fps
        if save:
            anim.save(f"{title}.gif", writer=PillowWriter(fps=frames/10))

        plt.close(self.fig)

        return HTML(anim.to_jshtml())